# 06. Text-to-SQL 맛보기
> Day 1 · 7H · 소요 약 50분

## 학습 목표

- LlamaIndex 의 `SQLDatabase` 래퍼를 이해하고 Neon DB 를 연결한다.
- `NLSQLTableQueryEngine` 으로 자연어 질문을 SQL 로 변환·실행한다.
- LLM 에 전달되는 `table_info` 의 실제 모습을 확인한다.
- "기본 설정만으로는 부족한" 실패 사례를 체험해 8H 심화로 가는 동기를 만든다.

> **선행 조건:** `01_postgres_basics.ipynb` 에서 적재한 병원 DB 가 필요합니다. `03_schema_intelligence.ipynb` 에서 추가한 `COMMENT ON` 이 있으면 `table_info` 품질이 더 좋아집니다.

In [ ]:
%pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai \
    sqlalchemy psycopg2-binary pandas

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")


## Text-to-SQL 작동 원리

```
자연어 질문 → [스키마 수집 → 프롬프트 조립 → LLM → SQL 생성 → SQL 실행 → 결과 요약] → 답변
```

**핵심은 `table_info`** — LLM 이 스키마를 이해하는 **유일한 정보원**입니다. `CREATE TABLE` + `COMMENT` + 샘플 행 이 합쳐진 텍스트이며, 이 품질이 정답률을 좌우합니다.

In [ ]:
# 이 노트북의 주인공은 LlamaIndex 의 SQLDatabase + NLSQLTableQueryEngine 두 클래스입니다.
# - SQLDatabase: SQLAlchemy 엔진을 한 번 더 감싸 LLM 친화적인 "스키마 텍스트(table_info)" 를 자동으로 만들어 줍니다.
# - NLSQLTableQueryEngine: 자연어 질문을 받아 → 프롬프트 조립 → LLM → SQL → 실행 → 답변까지 한 번에 합니다.
from sqlalchemy import create_engine
from llama_index.core import SQLDatabase, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

engine = create_engine(os.environ["NEON_DSN"])

# 모든 LlamaIndex 호출이 이 두 모델을 공통으로 사용하도록 전역 Settings 에 박아 둔다.
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# include_tables 로 LLM 에 노출할 테이블을 명시 — 이 리스트에 없는 테이블은 스키마 자체가 보이지 않습니다.
# 이렇게 화이트리스트를 두면 안전성과 토큰 비용을 동시에 잡을 수 있습니다.
sql_db = SQLDatabase(
    engine,
    include_tables=["patients", "doctors", "visits", "diagnoses", "departments"],
)
print("SQLDatabase ready.")

In [ ]:
# 연결된 테이블 목록 확인
print("Usable tables:", sql_db.get_usable_table_names())

In [ ]:
# NLSQLTableQueryEngine = "Natural Language SQL Table Query Engine" — 가장 단순한 Text-to-SQL 엔진.
# 내부 동작: 질문이 들어오면 (1) 스키마 텍스트(table_info) 와 함께 LLM 에 프롬프트 작성 →
# (2) LLM 이 SQL 생성 → (3) SQLDatabase 가 실제 실행 → (4) 결과를 다시 LLM 으로 자연어 답변.
from llama_index.core.query_engine import NLSQLTableQueryEngine

nlq = NLSQLTableQueryEngine(
    sql_database=sql_db,
    # 이 엔진이 사용할 수 있는 테이블을 한 번 더 명시 — 이 안에서만 SQL 이 만들어집니다.
    tables=["patients", "doctors", "visits", "diagnoses", "departments"],
)
print("Query engine ready.")

In [ ]:
# 성공 사례 시연: 3 가지 질문을 차례로 던지고 답변·SQL 을 함께 출력합니다.
# 핵심은 응답 객체의 .metadata["sql_query"] 를 항상 같이 보는 습관 — 이게 디버깅의 첫걸음입니다.
sample_questions = [
    "진료과별 의사 수를 알려주세요.",
    "환자 '홍길동'의 방문 이력을 보여주세요.",
    "2026년 1월의 방문 건수는?",
]
for q in sample_questions:
    resp = nlq.query(q)         # 한 줄로 자연어 → SQL → 결과 → 자연어 답변까지 처리
    print(f"Q: {q}")
    print(f"A: {resp.response}")
    # `.metadata` 는 dict. .get() 으로 키가 없을 때 빈 문자열을 기본값으로 받는다.
    print(f"SQL: {resp.metadata.get('sql_query','')}\n")

## 생성된 SQL 확인

응답 객체의 `metadata["sql_query"]` 에 실제 생성된 SQL 이 실립니다. 이 값을 보는 습관은 디버깅·프롬프트 튜닝에 가장 중요한 첫걸음입니다.

In [ ]:
resp = nlq.query("완료된 진료 중 진료비가 가장 높은 상위 5건은?")
print("Answer:\n", resp.response)
print("\nGenerated SQL:\n", resp.metadata.get("sql_query", "(no sql)"))

## `table_info` 실제로 들여다보기

LLM 이 실제로 보는 텍스트입니다. 여기에 `COMMENT` 가 잘 실려 있어야 LLM 이 "status = 'completed' 여야 진료가 실제 발생한 것" 같은 도메인 규칙을 올바르게 반영합니다.

In [ ]:
info = sql_db.get_single_table_info("visits")
print(info)

## 난이도별 질의 + 성공/실패 집계

Easy / Medium / Hard 로 분포시켜 결과를 기록합니다. 실패 사례를 모아두어야 뒤에서 프롬프트 개선의 **기준선(baseline)** 을 비교할 수 있습니다.

In [ ]:
test_questions = [
    ("Easy",   "남성 환자는 몇 명인가요?"),
    ("Easy",   "내과에 소속된 의사 목록을 보여주세요."),
    ("Easy",   "2026년 1월에 방문한 환자의 이름을 알려주세요."),
    ("Medium", "진료과별 의사 수를 알려주세요."),
    ("Medium", "완료된 진료 중 진료비가 가장 높은 상위 5건은?"),
    ("Medium", "2번 이상 방문한 환자의 이름과 방문 횟수를 보여주세요."),
    ("Hard",   "각 진료과별로 가장 최근에 진료한 의사의 이름은?"),
    ("Hard",   "월별 방문 추이를 전월 대비 증감과 함께 보여주세요."),
]

import pandas as pd
results = []
for level, question in test_questions:
    try:
        r = nlq.query(question)
        sql = r.metadata.get("sql_query", "")
        results.append({"level": level, "question": question, "status": "ok", "sql": sql})
    except Exception as e:
        results.append({"level": level, "question": question, "status": f"error: {type(e).__name__}", "sql": ""})

df = pd.DataFrame(results)
print(df[["level", "question", "status"]].to_string(index=False))
print(f"\nSuccess rate: {sum(r['status']=='ok' for r in results)}/{len(results)}")

## 기본 설정의 한계 — 모호한 질문은 실패한다

다음 질문들은 "자연어로는 자연스럽지만" 스키마만으로는 **정답을 특정할 수 없습니다.** LLM 은 임의로 해석을 선택해 틀린/엉뚱한 SQL 을 만들 수 있습니다.

In [ ]:
ambiguous_questions = [
    "최근에 많이 온 사람은?",        # '최근'은 언제? '많이'는 몇 번?
    "비싼 진료를 받은 환자는?",       # '비싸다'의 기준?
    "젊은 환자가 주로 가는 과는?",    # '젊다'의 기준?
]
for q in ambiguous_questions:
    try:
        r = nlq.query(q)
        print(f"Q: {q}")
        print(f"SQL: {r.metadata.get('sql_query','')}")
        print(f"A:   {r.response}\n")
    except Exception as e:
        print(f"Q: {q}\nerror: {e}\n")

### 왜 이런 일이 생기는가?

- `table_info` 에 **도메인 규칙**(예: "최근 = 지난 30일", "비싼 = 상위 10%")이 없음
- LLM 이 **자의적으로 해석**하여 SQL 을 만들어냄

→ **다음 노트북(08 · Day 2 10H)** 에서 프롬프트에 도메인 규칙을 주입하고 Few-shot 예제를 더해 정답률을 끌어올리는 방법을 배웁니다.

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 본인 질문 5개 테스트

**본인이 만든 질문 5개로 테스트하고 성공/실패를 기록하세요.**

_힌트: Easy/Medium/Hard 난이도가 섞이도록 질문 5개를 리스트로 만들고, 각 질문에 대해 `nlq.query(q)`를 `try/except`로 감싸서 `resp.metadata['sql_query']`와 `resp.response`를 출력하세요. 실패 시 예외 메시지를 같이 기록해 두면 분석에 유용합니다._

**성공/실패를 기록할 때 다음을 분석하세요:**

- 실패한 질문이 모호한 표현을 포함하고 있었나?
- 생성된 SQL에 잘못된 컬럼명이나 테이블명이 있었나?
- JOIN이 필요한 질문인데 단일 테이블만 조회했나?


In [ ]:
# ============================================================
# 실습 과제 — 본인 질문 5개 Text-to-SQL 테스트
# ============================================================

# 실습 1: 본인 질문 5개로 nlq.query() 테스트
# TODO: Easy/Medium/Hard 난이도가 섞이도록 질문 5개를 리스트로 만들고,
#       각 질문에 대해 try/except로 nlq.query(q)를 호출해
#       resp.metadata['sql_query']와 resp.response를 출력하세요.
#       실패 시 예외 메시지도 같이 기록하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

Day 1 을 마무리하며, 다음 **`07_project_briefing.ipynb`** 에서 **최종 프로젝트** 를 정식으로 브리핑합니다. 본인 도메인 선정, 제안서 양식, 평가 루브릭, 스키마 자동 검증기를 한 번에 정리합니다. Text-to-SQL 심화는 Day 2 첫 노트북(`08_text_to_sql_advanced.ipynb`)에서 이어집니다.